# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### My Baseline Rule

The rule is intentionally simple and transparent.

Content with higher Google Search Console clicks and impressions, together with better search position, receives a higher score.

Reason Codes

- HIGH_PERFORMER: High clicks and good search position.
- GROWING_CONTENT: Moderate performance with growth potential.
- LOW_VISIBILITY: Low clicks and poor search position.
- CTR_OPPORTUNITY: High impressions but relatively low clicks.

Action Labels

- Protect
- Monitor
- Improve SEO
- Rewrite

In [1]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

In [2]:
%pip -q install duckdb huggingface_hub

In [3]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

In [4]:
REL = """
read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

In [5]:
print("REL exists:", "REL" in globals())
print("con exists:", "con" in globals())

REL exists: True
con exists: True


In [6]:
query = f"""
SELECT
    CASE
        WHEN gsc_clicks = 0 THEN 'No Click'
        WHEN gsc_clicks BETWEEN 1 AND 10 THEN 'Low'
        WHEN gsc_clicks BETWEEN 11 AND 50 THEN 'Medium'
        ELSE 'High'
    END AS click_bucket,

    COUNT(*) AS n,
    ROUND(AVG(gsc_impressions),2) AS avg_impressions,
    ROUND(AVG(gsc_sum_position),2) AS avg_position

FROM {REL}

WHERE
    gsc_data_available IS TRUE

GROUP BY click_bucket

ORDER BY
CASE click_bucket
    WHEN 'No Click' THEN 1
    WHEN 'Low' THEN 2
    WHEN 'Medium' THEN 3
    WHEN 'High' THEN 4
END
"""

signal1 = con.sql(query).df()

signal1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,click_bucket,n,avg_impressions,avg_position
0,No Click,3193080,46.14,639.87
1,Low,412711,293.60,2762.89
2,Medium,5087,2077.79,11167.37
3,High,183,8728.06,33797.90


In [7]:
con.sql(f"""
SELECT *
FROM {REL}
LIMIT 1
""").df().columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc',
       'client_has_ga4', 'gsc_data_available', 'ga4_data_available',
       'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position',
       'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions',
       'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct',
       'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai',
       'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude',
       'ai_meta', 'ai_other', 'scroll_events', 'month'],
      dtype='object')

In [8]:
query = f"""
SELECT
CASE
    WHEN gsc_clicks = 0 THEN 'No Click'
    WHEN gsc_clicks BETWEEN 1 AND 10 THEN 'Low'
    WHEN gsc_clicks BETWEEN 11 AND 50 THEN 'Medium'
    ELSE 'High'
END AS click_bucket,

COUNT(*) AS n,

ROUND(AVG(gsc_impressions),2) AS avg_impressions,

ROUND(AVG(gsc_avg_position),2) AS avg_position

FROM {REL}

WHERE
gsc_data_available IS TRUE

GROUP BY click_bucket

ORDER BY
CASE click_bucket
WHEN 'No Click' THEN 1
WHEN 'Low' THEN 2
WHEN 'Medium' THEN 3
WHEN 'High' THEN 4
END
"""

signal1 = con.sql(query).df()

signal1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,click_bucket,n,avg_impressions,avg_position
0,No Click,3193080,46.14,16.75
1,Low,412711,293.60,8.79
2,Medium,5087,2077.79,5.20
3,High,183,8728.06,4.10


## Signal Check 1

**Signal:** Google Search Console Clicks

**Verdict:** **CONFIRMED**

The results show a clear relationship between clicks, impressions, and search position. Content with higher click counts also receives higher impressions and better average search positions. Therefore, Google Search Console clicks are a useful signal for identifying different content performance archetypes and can be safely used in the baseline rule.

In [9]:
query = f"""
SELECT
CASE
    WHEN gsc_avg_position <= 5 THEN 'Top 5'
    WHEN gsc_avg_position <= 10 THEN 'Top 10'
    WHEN gsc_avg_position <= 20 THEN 'Top 20'
    ELSE 'Beyond 20'
END AS position_bucket,

COUNT(*) AS n,

ROUND(AVG(gsc_clicks),2) AS avg_clicks,

ROUND(AVG(gsc_impressions),2) AS avg_impressions

FROM {REL}

WHERE gsc_data_available IS TRUE

GROUP BY position_bucket

ORDER BY
CASE position_bucket
WHEN 'Top 5' THEN 1
WHEN 'Top 10' THEN 2
WHEN 'Top 20' THEN 3
WHEN 'Beyond 20' THEN 4
END;
"""

signal2 = con.sql(query).df()

signal2

,position_bucket,n,avg_clicks,avg_impressions
0,Top 5,1263125,0.35,96.51
1,Top 10,920359,0.22,76.01
2,Top 20,519223,0.18,56.60
3,Beyond 20,908354,0.09,65.41


## Signal Check 2

**Signal:** Google Search Console Average Position

**Verdict:** **CONFIRMED**

The results indicate that better search positions are generally associated with higher clicks and stronger visibility. Content ranked in the Top 5 has the highest average clicks (0.35) and impressions (96.51), while clicks gradually decrease as the average position becomes worse (Top 10 → Top 20 → Beyond 20). Although the "Beyond 20" bucket shows slightly higher average impressions than the "Top 20" bucket, it still receives substantially fewer clicks, suggesting lower search effectiveness. Therefore, average search position is confirmed as a reliable signal for identifying different content performance archetypes and is suitable for the baseline scoring rule.

# 2. Build the Ranked Queue

## Baseline Rule

A content item is considered a stronger content archetype when it receives high search visibility, generates more clicks, and maintains a better average search position.

The baseline score combines these three signals to rank content into meaningful performance archetypes.

Each ranked content item receives:
- A baseline score
- A reason code
- An action label

In [10]:
import os

query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,

    (
        (gsc_impressions * 0.01) +
        (gsc_clicks * 5) +
        ((21 - LEAST(gsc_avg_position,20))*3)
    ) AS baseline_score,

    CASE
        WHEN gsc_clicks >= 50
             AND gsc_avg_position <=5
            THEN 'High Visibility'

        WHEN gsc_clicks >=10
             AND gsc_avg_position <=10
            THEN 'Growing Content'

        ELSE 'Low Visibility'
    END AS reason_code,

    CASE
        WHEN gsc_clicks >=50
             AND gsc_avg_position<=5
            THEN 'Protect'

        WHEN gsc_clicks>=10
             AND gsc_avg_position<=10
            THEN 'Monitor'

        ELSE 'Improve'
    END AS action

FROM {REL}

WHERE gsc_data_available IS TRUE

ORDER BY baseline_score DESC
"""

baseline = con.sql(query).df()

baseline.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,baseline_score,reason_code,action
0,2026-03-28,client_e547b89c05043229,content_eadb33b5df496f4a,38436,271,2.195988,1795.772036,High Visibility,Protect
1,2026-03-29,client_e547b89c05043229,content_eadb33b5df496f4a,39305,252,2.197507,1709.457480,High Visibility,Protect
2,2026-03-15,client_e547b89c05043229,content_eadb33b5df496f4a,16706,274,2.357117,1592.988648,High Visibility,Protect
3,2026-03-31,client_e547b89c05043229,content_eadb33b5df496f4a,34606,235,2.242501,1577.332496,High Visibility,Protect
4,2026-03-30,client_e547b89c05043229,content_eadb33b5df496f4a,35404,225,2.188397,1535.474810,High Visibility,Protect


In [11]:
os.makedirs("work/outputs", exist_ok=True)

baseline.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully.")
print("Rows:", len(baseline))

CSV saved successfully.
Rows: 3611061


In [12]:
baseline.head(20)

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,baseline_score,reason_code,action
0,2026-03-28,client_e547b89c05043229,content_eadb33b5df496f4a,38436,271,2.195988,1795.772036,High Visibility,Protect
1,2026-03-29,client_e547b89c05043229,content_eadb33b5df496f4a,39305,252,2.197507,1709.457480,High Visibility,Protect
2,2026-03-15,client_e547b89c05043229,content_eadb33b5df496f4a,16706,274,2.357117,1592.988648,High Visibility,Protect
3,2026-03-31,client_e547b89c05043229,content_eadb33b5df496f4a,34606,235,2.242501,1577.332496,High Visibility,Protect
4,2026-03-30,client_e547b89c05043229,content_eadb33b5df496f4a,35404,225,2.188397,1535.474810,High Visibility,Protect
5,2026-03-13,client_e547b89c05043229,content_eadb33b5df496f4a,16462,260,2.440165,1520.299504,High Visibility,Protect
6,2026-03-27,client_e547b89c05043229,content_eadb33b5df496f4a,34817,223,2.181348,1519.625955,High Visibility,Protect
7,2026-03-14,client_e547b89c05043229,content_eadb33b5df496f4a,13515,264,2.252090,1511.393729,High Visibility,Protect
8,2026-03-31,client_23a62021009f63c4,content_e6df0936699f5b8f,14682,269,25.035826,1494.820000,Low Visibility,Improve
9,2026-03-22,client_e547b89c05043229,content_eadb33b5df496f4a,32665,221,2.388857,1487.483430,High Visibility,Protect


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [13]:
top20 = baseline.head(20)

top20

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,baseline_score,reason_code,action
0,2026-03-28,client_e547b89c05043229,content_eadb33b5df496f4a,38436,271,2.195988,1795.772036,High Visibility,Protect
1,2026-03-29,client_e547b89c05043229,content_eadb33b5df496f4a,39305,252,2.197507,1709.457480,High Visibility,Protect
2,2026-03-15,client_e547b89c05043229,content_eadb33b5df496f4a,16706,274,2.357117,1592.988648,High Visibility,Protect
3,2026-03-31,client_e547b89c05043229,content_eadb33b5df496f4a,34606,235,2.242501,1577.332496,High Visibility,Protect
4,2026-03-30,client_e547b89c05043229,content_eadb33b5df496f4a,35404,225,2.188397,1535.474810,High Visibility,Protect
5,2026-03-13,client_e547b89c05043229,content_eadb33b5df496f4a,16462,260,2.440165,1520.299504,High Visibility,Protect
6,2026-03-27,client_e547b89c05043229,content_eadb33b5df496f4a,34817,223,2.181348,1519.625955,High Visibility,Protect
7,2026-03-14,client_e547b89c05043229,content_eadb33b5df496f4a,13515,264,2.252090,1511.393729,High Visibility,Protect
8,2026-03-31,client_23a62021009f63c4,content_e6df0936699f5b8f,14682,269,25.035826,1494.820000,Low Visibility,Improve
9,2026-03-22,client_e547b89c05043229,content_eadb33b5df496f4a,32665,221,2.388857,1487.483430,High Visibility,Protect


In [14]:
top20_review = top20[
    [
        "content_hash_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
]

top20_review

,content_hash_id,baseline_score,reason_code,action
0,content_eadb33b5df496f4a,1795.772036,High Visibility,Protect
1,content_eadb33b5df496f4a,1709.457480,High Visibility,Protect
2,content_eadb33b5df496f4a,1592.988648,High Visibility,Protect
3,content_eadb33b5df496f4a,1577.332496,High Visibility,Protect
4,content_eadb33b5df496f4a,1535.474810,High Visibility,Protect
5,content_eadb33b5df496f4a,1520.299504,High Visibility,Protect
6,content_eadb33b5df496f4a,1519.625955,High Visibility,Protect
7,content_eadb33b5df496f4a,1511.393729,High Visibility,Protect
8,content_e6df0936699f5b8f,1494.820000,Low Visibility,Improve
9,content_eadb33b5df496f4a,1487.483430,High Visibility,Protect


In [ ]:
# 3. Top-20 Review

The ranked queue was reviewed by checking the top 20 ranked content items.

### Review Summary

1. Most of the top-ranked items have high impressions, high clicks, and good search positions, so the **Protect** action is reasonable.

2. Some top-ranked rows belong to the same content item on different report dates because the dataset contains one row per content per day.

3. One item (Rank 9) received the **Improve** action even though it has many clicks because its average search position is poor. This shows that the rule gives more importance to search position.

4. Overall, the baseline rule identifies high-performing content, but repeated daily records may affect the ranking.

### What could make the ranking wrong?

- Temporary traffic increases.
- Seasonal changes.
- Daily variations in performance.
- Duplicate daily records for the same content.

# 4. Weak Picks + Leakage Check

### Weak Picks

One weak pick was found during the review.

The content ranked at **Rank 9** was given the **Improve** action because its average search position was greater than 20. However, it also received a high number of clicks. This suggests that the current baseline rule may give too much importance to search position.

In future work, the scoring rule could be improved by balancing clicks, impressions, and search position more effectively.

### Leakage Check

No product flags or future-window information were used in the baseline score.

The score was calculated using only:
- GSC impressions
- GSC clicks
- GSC average position

All of these values are available at the decision time and do not use future information.

Therefore, no data leakage was introduced in this baseline.

# 4. Weak Picks + Leakage Check

### Weak Picks

One weak pick was found during the review.

The content ranked at **Rank 9** was given the **Improve** action because its average search position was greater than 20. However, it also received a high number of clicks. This suggests that the current baseline rule may give too much importance to search position.

In future work, the scoring rule could be improved by balancing clicks, impressions, and search position more effectively.

### Leakage Check

No product flags or future-window information were used in the baseline score.

The score was calculated using only:
- GSC impressions
- GSC clicks
- GSC average position

All of these values are available at the decision time and do not use future information.

Therefore, no data leakage was introduced in this baseline.

# 4. Weak Picks + Leakage Check

### Weak Picks

One weak pick was found during the review.

The content ranked at **Rank 9** was given the **Improve** action because its average search position was greater than 20. However, it also received a high number of clicks. This suggests that the current baseline rule may give too much importance to search position.

In future work, the scoring rule could be improved by balancing clicks, impressions, and search position more effectively.

### Leakage Check

No product flags or future-window information were used in the baseline score.

The score was calculated using only:
- GSC impressions
- GSC clicks
- GSC average position

All of these values are available at the decision time and do not use future information.

Therefore, no data leakage was introduced in this baseline.

✓ Every section above is filled.
✓ The notebook runs top to bottom with no errors.
✓ No client names, URLs, or private queries anywhere.
✓ My claims use careful words.
✓ Committed to my repo.